In [ ]:
# import required libs
import numpy as np
import sys
sys.path.append(r"../Communiaction")
sys.path.append(r"../Utils")
sys.path.append(r"../../NeuralNetworks/Utils")
import TCP_IP_UTILS
import NN_Utils
import Utils
import time
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch
import torch.nn as nn

In [ ]:
#Add power supply code here -> DC Sources
import sys
import pyvisa
sys.path.append(r"../PowerSupply")
import PS_Utils
Device_ID_DC = "USB0::0x2A8D::0x1202::MY59003914::0::INSTR"
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("Available VISA resources:")
for r in resources:
        print(" -", r)
device_dc = PS_Utils.connect_E36313A(Device_ID_DC)
DIG_LS = 1
ANA_LS = 2
MAIN = 3

In [ ]:
#connect to ethernet socket 
HOST = '192.168.1.10'  # Remote device IP (server IP)
PORT = 7               # Echo port
client_socket = TCP_IP_UTILS.tcp_connect(HOST,PORT)
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)
time.sleep(1) #Wait for IOs to settle

In [ ]:
#First Turn on Digital Leval Shifters
PS_Utils.set_voltage_E36313A(device_dc,DIG_LS,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,DIG_LS))
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

In [ ]:
#Turn on Main Chip PS
PS_Utils.set_voltage_E36313A(device_dc,MAIN,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,MAIN))

In [ ]:
#Turn on CLK 
#Device_ID_CLK = "USB0::0x0957::0x1F01::MY57280340::0::INSTR"
Device_ID_CLK = "USB0::0x0957::0x1F01::MY53270560::0::INSTR"
device_clk = PS_Utils.connect_N5173B(Device_ID_CLK)
PS_Utils.set_voltage_N5173B(device_clk,0.45)
PS_Utils.set_frequency_N5173B(device_clk,1000000000)
PS_Utils.enable_output_N5173B(device_clk)

In [ ]:
#Call DL EN Scan Chain here with data
scan_id = Utils.DL_EN_SC_LM["id"]
scan_len = Utils.DL_EN_SC_LM["len"]
scan_data = np.ones(scan_len,dtype=int).tolist()
#scan_data[4] = 0
#scan_data[0] = 0
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)

In [ ]:
#Call DL EN Scan Chain here with data
scan_id = Utils.DL_EN_SC_RM["id"]
scan_len = Utils.DL_EN_SC_RM["len"]
scan_data = np.ones(scan_len,dtype=int).tolist()
#scan_data[71] = 0
#scan_data[4] = 0
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)

In [ ]:
#Control Logic TD Data
import numpy as np
import random
import re
def scan_gen(time):
    SC_data = f"""
    IN_EXT_LOC = 1'b0;
    TDC_EN_EXT_LOC = 1'b0;
    ENCHG_EXT_LOC = 1'b0;
    ENTIME_EXT_LOC = 1'b1;
    PCHG_EXT_LOC = 1'b0;
    VDAC_CTRL_EXT_LOC = 1'b0;
    DEL_RST_EXT_LOC = 1'b0;
    TDC_COMPUTE_EXT_LOC = 1'b0;
    TDC_RST_EXT_LOC = 1'b0;
    VTC_EN_EXT_LOC = 1'b0;

    IN_LB = 6'd3;
    IN_UB = 6'd22; 
    TDC_EN_LB = 6'd1;
    TDC_EN_UB = 6'd21;
    ENCHG_LB = 6'd0;
    ENCHG_UB = 6'd0;
    ENTIME_LB = 6'd0;
    ENTIME_UB = 6'd0;
    PCHG_LB = 6'd2;
    PCHG_UB = 6'd22;
    VDAC_CTRL_LB = 6'd0;
    VDAC_CTRL_UB = 6'd0; 
    DEL_RST_LB = 6'd2;
    DEL_RST_UB = 6'd22;
    TDC_COMPUTE_LB = 6'd19;
    TDC_COMPUTE_UB = 6'd24;
    TDC_RST_LB = 6'd1;
    TDC_RST_UB = 6'd21;
    BL_NUM_TGT = 2'd3;
    BL_DONE_TIME = 6'd55;
    VTC_EN_LB = 6'd0;
    VTC_EN_UB = 6'd0;
    SAMPLE_EDGE_TIME = 6'd{time};

    CHG_TIME = 1'd0;
    OSC_DEL = 1'd0;
    """
    SC_order = ["BL_DONE_TIME","SAMPLE_EDGE_TIME", "BL_NUM_TGT", "BLANK_4", "IN_LB", "TDC_EN_LB", "ENCHG_LB", "IN_EXT_LOC","TDC_EN_EXT_LOC","ENCHG_EXT_LOC",\
                "ENTIME_EXT_LOC","PCHG_EXT_LOC","VDAC_CTRL_EXT_LOC", 'ENTIME_LB', 'PCHG_LB', 'VDAC_CTRL_LB', 'DEL_RST_LB', 'TDC_COMPUTE_LB', 'TDC_RST_LB', \
                'VTC_EN_LB',"DEL_RST_EXT_LOC","TDC_COMPUTE_EXT_LOC","TDC_RST_EXT_LOC","VTC_EN_EXT_LOC","CHG_TIME","OSC_DEL", "VTC_EN_UB","TDC_RST_UB","TDC_COMPUTE_UB",\
                "DEL_RST_UB","VDAC_CTRL_UB","PCHG_UB","ENTIME_UB","ENCHG_UB","TDC_EN_UB","IN_UB","BLANK_12_PISO"]

    lines = SC_data.splitlines()
    sc_signal = {}
    for line in lines:
        if line.strip():  # Skip empty lines
            # Handle 'd' (decimal) format
            match = re.match(r"(\w+) = (\d+)'d([0-9]+);", line.strip())
            if match:
                signal_name = match.group(1)
                num_bits = int(match.group(2))  # Number of bits
                signal_value = match.group(3)  # Decimal value


                # If there's only 1 bit, do not add the bit-range
                if num_bits == 1:
                    name = signal_name
                else:
                    # Format the name with the bit-range (e.g., VTC_EN_UB<[5:0]>)
                    name = "%s" % (signal_name) # Using 0-based index for the range

                # Format the decimal value to binary and pad to the correct number of bits
                binary_value = bin(int(signal_value))[2:].zfill(num_bits)
                sc_signal[name] = binary_value


            # Handle 'b' (binary) format
            elif "b" in line:  # e.g., BL_NUM_TGT = 2'b01;
                match = re.match(r"(\w+) = (\d+)'b([01]+);", line.strip())
                if match:
                    signal_name = match.group(1)
                    num_bits = int(match.group(2))  # Number of bits
                    signal_value = match.group(3)  # Binary value

                    # If there's only 1 bit, do not add the bit-range
                    if num_bits == 1:
                        name = signal_name
                    else:
                        # Format the name with the bit-range (e.g., BL_NUM_TGT<[1:0]> for 2 bits)
                        name = "%s" % (signal_name)

                    # Format the binary value to match the number of bits
                    binary_value = signal_value.zfill(num_bits)
                    sc_signal[name] = binary_value


    # print(sc_signal)
    output_list = []
    for signal in SC_order:
        if "BLANK" in signal:
            num_blanks = re.findall(r'\d+', signal)
            output_list += [0]*int(num_blanks[0])
        else:
            temp = sc_signal[signal][::-1]
            output_list += list(map(int,temp))
    print(sc_signal["SAMPLE_EDGE_TIME"])
    output_list = output_list[::-1] + [0]*30
    return output_list

def set_CS(cs):
    ret_data = 0
    if cs == 0:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [0,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 1:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [1,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 2:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [0,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 3:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [1,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    return ret_data

In [ ]:
#Call Scan IN here with data
scan_id = Utils.CONTROL_LOGIC_SC["id"]
scan_len = Utils.CONTROL_LOGIC_SC["len"]
scan_data = scan_gen(5)
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
#print(ret_data)

In [ ]:
#Initialise SRAM with default data.
path = r"..\Utils\CalibrationData.xlsx"
write_data_default = Utils.get_Default_Write_Data(path).tolist()
ret_data = Utils.Write_SRAM(client_socket,write_data_default,1)
print(ret_data)

In [ ]:
#Turn on Analog PS
PS_Utils.set_voltage_E36313A(device_dc,ANA_LS,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,ANA_LS))

In [ ]:
flattened = (Utils.TD_CORRECTION).flatten()
ret_data = Utils.Send_CorrectionData(client_socket,flattened,0)
print(ret_data)

In [ ]:
x = torch.tensor([[1.1, 2.344, 3.56],
                  [4.0, 5.0, 6.0]])

# Define Linear layer: 3 input features → 2 output features, no bias
linear = nn.Linear(in_features=3, out_features=2, bias=False)

# Manually set weights for clarity (optional)
# Shape of weight: (out_features, in_features)
linear.weight.data = torch.tensor([[10.0, 10.0, 1.0],
                                   [10.5, 1.5, 0.5]])
# Apply linear layer
output = linear(x)

# Print result
print("Output:\n", output)

In [ ]:
TD_DEL_LUT_PATH = r"../../NeuralNetworks/Utils/Hardware_Model/HW_MODEL_TD_DEL.npy"
TD_LUT_PATH = r"../../NeuralNetworks/Utils/Hardware_Model/HW_MODEL_TD_FastIMC.npy" #r"../Utils/Hardware_Model/HW_MODEL_TD_1x_Ref256.npy"
CD_LUT_PATH = r"../../NeuralNetworks/Utils/Hardware_Model/HW_MODEL_CD_FastIMC.npy" #r"../Utils/Hardware_Model/HW_MODEL_CD_4x_Ref256_sweep4.npy"
LUT_PATHS = [TD_LUT_PATH,CD_LUT_PATH]
LUT_FLAGS = [False,True,True]

In [ ]:
Lin1 = NN_Utils.Linear(0,x,linear.weight.data,[8,10],None,client_socket,LUT_FLAGS,LUT_PATHS)
out,sum = Lin1.linear_onChip()
print("Output:\n", out)

In [ ]:
Lin1 = NN_Utils.Linear(1,x,linear.weight.data,[8,10],None,client_socket,LUT_FLAGS,LUT_PATHS)
out,sum = Lin1.linear_onChip()
print("Output:\n", out)

In [ ]:
#Call Read SRAM here
SRAM_DATA = Utils.Read_SRAM(client_socket,1)
print(len(SRAM_DATA))
print(SRAM_DATA)

In [ ]:
#Set IO to default and exit    
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

In [ ]:
PS_Utils.kill_channel_E36313A(device_dc,ANA_LS)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,MAIN)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,DIG_LS)
PS_Utils.disconnect_E36313A(device_dc)
time.sleep(0.3)
PS_Utils.kill_N5173B(device_clk)
PS_Utils.disconnect_N5173B(device_clk)